In [49]:
# Instalar PySpark
!pip install -qqq pyspark

# Verificar Java
!java -version

# Configurar variables de entorno (si fueran necesarias)
import os

os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [50]:
!pip install -q pyspark

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("LaboratorioBigData")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

In [51]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
ruta='/content/drive/MyDrive/Grupo4_CIIN1021P_EP/'
file1='RENIPRESS_31-08-2026.csv'
file2='atenciones_la_libertad_2024s2.csv'

In [53]:
# ============================================================
# CARGA DE DATOS
# spark.read.csv devuelve un DataFrame, es decir, df_ren y df_aten
# ya son objetos DataFrame desde este punto.
# ============================================================

# spark.read.csv devuelve un DataFrame: df_ren y df_aten ya son DataFrames.

# RENIPRESS: separador ';' y codificación iso-8859-1
df_ren = spark.read.csv(ruta+file1, header=True, inferSchema=True,
                        sep=';', encoding='iso-8859-1')

# Atenciones SIS: separador ',' y codificación utf-8
df_aten = spark.read.csv(ruta+file2, header=True, inferSchema=True,
                         sep=',', encoding='utf-8')

# Se renombra AÑO para evitar problemas con la Ñ en las consultas
df_aten = df_aten.withColumnRenamed('AÑO', 'ANIO')

print(df_aten.columns)                    # deben salir las 17 columnas separadas
print(df_ren.count(), df_aten.count())    # 36005 y 256448

['ANIO', 'MES', 'REGION', 'PROVINCIA', 'UBIGEO_DISTRITO', 'DISTRITO', 'COD_UNIDAD_EJECUTORA', 'DESC_UNIDAD_EJECUTORA', 'COD_IPRESS', 'IPRESS', 'NIVEL_EESS', 'PLAN_SEGURO', 'COD_SERVICIO', 'DESC_SERVICIO', 'SEXO', 'GRUPO_EDAD', 'ATENCIONES']
36005 256448


In [54]:
# ============================================================
# LIMPIEZA CON DATAFRAME (funciones de pyspark.sql.functions)
# ============================================================
from pyspark.sql import functions as F

# DATAFRAME: se elimina la fila sin código y se normaliza COD_IPRESS
# a 8 dígitos con ceros a la izquierda (mismo criterio que en SQL Server)
df_ren = (df_ren.dropna(subset=['COD_IPRESS'])
                .withColumn('COD_IPRESS',
                            F.lpad(F.col('COD_IPRESS').cast('string'), 8, '0')))

df_aten = df_aten.withColumn('COD_IPRESS',
                             F.lpad(F.col('COD_IPRESS').cast('string'), 8, '0'))

# Validación: los conteos deben coincidir con los de SQL Server
print("RENIPRESS:", df_ren.count())                                               # 36004
print("Atenciones que cruzan:",
      df_aten.join(df_ren.select('COD_IPRESS'), 'COD_IPRESS').count())            # 256448
print("Establecimientos distintos:",
      df_aten.select('COD_IPRESS').distinct().count())                            # 335

RENIPRESS: 36005
Atenciones que cruzan: 256448
Establecimientos distintos: 335


In [55]:
# Con datos pequeños, 8 particiones en los shuffles son suficientes
# (el valor por defecto es 200 y agrega trabajo innecesario)
spark.conf.set("spark.sql.shuffle.partitions", "8")

# cache() deja los DataFrames en memoria; el count() fuerza la carga una vez.
# Así los tiempos miden la consulta y no la lectura de los CSV desde Drive.
df_aten = df_aten.cache()
df_ren = df_ren.cache()
print(df_aten.count(), df_ren.count())

256448 36005


In [56]:
# ============================================================
# ESTADÍSTICA DESCRIPTIVA CON DATAFRAME
# ============================================================

# DATAFRAME: resumen estadístico de la columna ATENCIONES
df_aten.select('ATENCIONES').summary(
    'count', 'mean', 'stddev', 'min', '25%', '50%', '75%', 'max').show()

# DATAFRAME: agregación (groupBy + sum) por grupo de edad
(df_aten.groupBy('GRUPO_EDAD')
        .agg(F.sum('ATENCIONES').alias('total'))
        .orderBy(F.desc('total'))
        .show())

+-------+------------------+
|summary|        ATENCIONES|
+-------+------------------+
|  count|            256448|
|   mean| 9.705577738956826|
| stddev|25.087367251110162|
|    min|                 1|
|    25%|                 1|
|    50%|                 3|
|    75%|                 9|
|    max|              1166|
+-------+------------------+

+-------------+------+
|   GRUPO_EDAD| total|
+-------------+------+
| 00 - 04 AÑOS|640864|
| 30 - 59 AÑOS|622302|
| 18 - 29 AÑOS|371179|
|60 - MAS AÑOS|352858|
| 05 - 11 AÑOS|257684|
| 12 - 17 AÑOS|244089|
+-------------+------+



In [57]:
# ============================================================
# SPARKSQL
# Se registran los DataFrames como vistas temporales para poder
# consultarlos con SQL estándar mediante spark.sql().
# ============================================================
df_aten.createOrReplaceTempView("atenciones")
df_ren.createOrReplaceTempView("renipress")

# SPARKSQL: atenciones por provincia y categoría del establecimiento
q = """
SELECT r.PROVINCIA, r.CATEGORIA,
       SUM(a.ATENCIONES) AS total_atenciones,
       COUNT(*)          AS n_registros
FROM atenciones a
JOIN renipress r ON a.COD_IPRESS = r.COD_IPRESS
GROUP BY r.PROVINCIA, r.CATEGORIA
ORDER BY total_atenciones DESC
"""
spark.sql(q).show(10, truncate=False)

+---------------+---------+----------------+-----------+
|PROVINCIA      |CATEGORIA|total_atenciones|n_registros|
+---------------+---------+----------------+-----------+
|TRUJILLO       |II-1     |242995          |13795      |
|SANCHEZ CARRION|I-2      |203563          |22763      |
|TRUJILLO       |I-3      |186823          |18204      |
|TRUJILLO       |I-4      |155831          |12382      |
|TRUJILLO       |I-2      |147953          |17299      |
|TRUJILLO       |III-1    |96722           |3026       |
|OTUZCO         |I-2      |82227           |12257      |
|TRUJILLO       |III-2    |72321           |1381       |
|PACASMAYO      |II-1     |67634           |4363       |
|SANCHEZ CARRION|II-1     |66399           |2105       |
+---------------+---------+----------------+-----------+
only showing top 10 rows


In [58]:
# ============================================================
# DATAFRAME API
# Misma consulta que la celda anterior, escrita con métodos de
# DataFrame (join, groupBy, agg, orderBy) en lugar de SQL.
# ============================================================
def consulta_df(df_a=df_aten):
    # DATAFRAME: se toma solo lo necesario de RENIPRESS y se renombra
    # PROVINCIA para evitar la referencia ambigua con df_aten
    ren = df_ren.select('COD_IPRESS',
                        F.col('PROVINCIA').alias('PROVINCIA_R'),
                        'CATEGORIA')

    return (df_a
            # DATAFRAME: join por COD_IPRESS
            .join(ren, 'COD_IPRESS')
            # DATAFRAME: agrupación y agregaciones
            .groupBy('PROVINCIA_R', 'CATEGORIA')
            .agg(F.sum('ATENCIONES').alias('total_atenciones'),
                 F.count('*').alias('n_registros'))
            .orderBy(F.desc('total_atenciones'))
            .withColumnRenamed('PROVINCIA_R', 'PROVINCIA'))

# El resultado debe coincidir con el de SparkSQL (celda 9)
consulta_df().show(10, truncate=False)

+---------------+---------+----------------+-----------+
|PROVINCIA      |CATEGORIA|total_atenciones|n_registros|
+---------------+---------+----------------+-----------+
|TRUJILLO       |II-1     |242995          |13795      |
|SANCHEZ CARRION|I-2      |203563          |22763      |
|TRUJILLO       |I-3      |186823          |18204      |
|TRUJILLO       |I-4      |155831          |12382      |
|TRUJILLO       |I-2      |147953          |17299      |
|TRUJILLO       |III-1    |96722           |3026       |
|OTUZCO         |I-2      |82227           |12257      |
|TRUJILLO       |III-2    |72321           |1381       |
|PACASMAYO      |II-1     |67634           |4363       |
|SANCHEZ CARRION|II-1     |66399           |2105       |
+---------------+---------+----------------+-----------+
only showing top 10 rows


In [63]:
import time
import statistics as st
# Ejecuta la función n veces y devuelve los tiempos y el promedio
# sin la primera corrida (que incluye el arranque de Spark)
def medir(fn, n=5):
    t = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()                                 # collect() fuerza la ejecución real
        t.append(time.perf_counter() - t0)
    resto = t[1:]
    return {"primera": t[0],
            "mediana": st.median(resto),
            "promedio": sum(resto)/len(resto),
            "desv": st.stdev(resto) if len(resto) > 1 else 0.0}
# SPARKSQL: tiempo de la consulta escrita en SQL
m_sql = medir(lambda: spark.sql(q).collect())

# DATAFRAME: tiempo de la misma consulta con la API de DataFrame
m_df = medir(lambda: consulta_df().collect())

for nombre, m in [("SparkSQL", m_sql), ("DataFrame", m_df)]:
    print(f"{nombre:10s} 1ª: {m['primera']:.3f}s | mediana: {m['mediana']:.3f}s | "
          f"promedio: {m['promedio']:.3f}s | desv: {m['desv']:.3f}s")

SparkSQL   1ª: 1.814s | mediana: 2.511s | promedio: 2.659s | desv: 0.772s
DataFrame  1ª: 2.817s | mediana: 2.064s | promedio: 1.955s | desv: 0.261s


In [64]:
# ============================================================
# PRUEBA DE ESCALABILIDAD (DATAFRAME)
# Se replica el dataset de atenciones con crossJoin contra un
# rango pequeño y se mide la misma consulta en cada volumen.
# ============================================================
import pandas as pd

def replicar(df, factor):
    return (df.crossJoin(spark.range(factor).withColumnRenamed("id", "_rep"))
              .drop("_rep"))

resultados = []
for factor in [1, 5, 10, 20, 40, 80]:
    df_x = replicar(df_aten, factor).cache()
    filas = df_x.count()                       # materializa en memoria antes de medir

    m = medir(lambda: consulta_df(df_x).collect(), n=4)
    resultados.append((factor, filas,
                       round(m["primera"], 3),
                       round(m["mediana"], 3),
                       round(m["desv"], 3)))
    print(resultados[-1])

    df_x.unpersist()                           # libera memoria para el siguiente factor

tabla = pd.DataFrame(resultados,
        columns=["Factor", "Filas", "Spark 1ª (s)", "Spark mediana (s)", "Spark desv"])
tabla

(1, 256448, 1.972, 1.015, 0.513)
(5, 1282240, 1.328, 1.173, 0.441)
(10, 2564480, 1.334, 1.218, 0.152)
(20, 5128960, 2.061, 2.387, 0.24)
(40, 10257920, 4.126, 4.76, 0.722)
(80, 20515840, 9.41, 7.212, 1.461)


,Factor,Filas,Spark 1ª (s),Spark mediana (s),Spark desv
0,1,256448,1.972,1.015,0.513
1,5,1282240,1.328,1.173,0.441
2,10,2564480,1.334,1.218,0.152
3,20,5128960,2.061,2.387,0.240
4,40,10257920,4.126,4.760,0.722
5,80,20515840,9.410,7.212,1.461


In [66]:
import pandas as pd

# Tiempos de SQL Server en segundos (mediana de las corridas 2 a 4, medidos en SSMS)
sqlserver = {1: 0.060, 5: 0.204, 10: 0.280, 20: 0.841, 40: 1.680, 80: 3.268}

tabla["SQL Server (s)"] = tabla["Factor"].map(sqlserver)
tabla["Razón SQL/Spark"] = (tabla["SQL Server (s)"] / tabla["Spark mediana (s)"]).round(2)
tabla

,Factor,Filas,Spark 1ª (s),Spark mediana (s),Spark desv,SQL Server (s),Razón SQL/Spark
0,1,256448,1.972,1.015,0.513,0.060,0.06
1,5,1282240,1.328,1.173,0.441,0.204,0.17
2,10,2564480,1.334,1.218,0.152,0.280,0.23
3,20,5128960,2.061,2.387,0.240,0.841,0.35
4,40,10257920,4.126,4.760,0.722,1.680,0.35
5,80,20515840,9.410,7.212,1.461,3.268,0.45
